# DALI v2 → MOHIM motif dataset → ACE-Step LoRA

기존 `MOHIM.ipynb`와 독립된 실행 노트북입니다. 먼저 1곡을 듣고, 다음으로 3곡 smoke test를 완료한 뒤 전체 데이터로 확장하세요.

## 0. 저장소와 패키지 준비

In [ ]:
from pathlib import Path
import os, subprocess, sys

MOHIM_REPOSITORY = "https://github.com/youhan200203/MOHIM.git"
MOHIM_BRANCH = "working"
REPO_DIR = Path("/content/MOHIM")

if not (REPO_DIR / "requirements-dali.txt").is_file():
    subprocess.run(
        ["git", "clone", "--branch", MOHIM_BRANCH, MOHIM_REPOSITORY, str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
print("repository:", Path.cwd())

In [ ]:
%pip install -q -r requirements-dali.txt

# 설치 후 import cache 문제를 피하려면 처음 한 번만 런타임을 재시작할 수 있습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. 경로와 임계값 설정

In [ ]:
from pathlib import Path

DALI_DATA_DIR = Path("/content/drive/MyDrive/MOHIM/dali_v2")
DALI_AUDIO_DIR = Path("/content/drive/MyDrive/MOHIM/dali_audio")
OUTPUT_DIR = Path("/content/drive/MyDrive/MOHIM/dali_motif_dataset")
MANIFEST_PATH = OUTPUT_DIR / "dual_stream_manifest.json"
BEAT_CHECKPOINT = Path("/content/checkpoints/beat_this_final0.ckpt")

DEVICE = "cuda"
AUDIO_FORMAT = "flac"       # wav보다 저장 공간 절약
MAX_SONGS = 3                  # 먼저 3곡으로 확인; 전체 실행은 None
MOTIF_BARS = 4
MOTIF_SIMILARITY = 0.56
MOTIF_SEARCH_SECONDS = 45.0

for path in (DALI_DATA_DIR, DALI_AUDIO_DIR):
    print(path, "exists=", path.exists())

## 2. DALI 로딩과 영어 Pop 필터

DALI의 `lines[].text`만 결합하므로 단어별 시간 정렬값은 학습 manifest에 들어가지 않습니다.

In [ ]:
from mohim_dali.dali import load_dali, filter_tracks

all_tracks = load_dali(DALI_DATA_DIR)
pop_tracks = filter_tracks(all_tracks, language="english", genre="pop", require_lyrics=True)
print("all DALI tracks:", len(all_tracks))
print("English Pop with lyrics:", len(pop_tracks))

In [ ]:
import pandas as pd

preview = pd.DataFrame([
    {"dali_id": t.dali_id, "artist": t.artist, "title": t.title,
     "genres": ", ".join(t.genres), "language": t.language, "lyrics_chars": len(t.lyrics)}
    for t in pop_tracks[:20]
])
display(preview)

In [ ]:
sample_track = pop_tracks[0]
print(sample_track.artist, "-", sample_track.title)
print("genres:", sample_track.genres)
print("\n--- plain lyrics preview ---")
print(sample_track.lyrics[:1500])

## 3. 로컬 음원 매칭 확인

In [ ]:
from mohim_dali.dali import index_audio_files, resolve_audio_path

audio_index = index_audio_files(DALI_AUDIO_DIR)
matched = [(track, resolve_audio_path(track, audio_index)) for track in pop_tracks]
matched = [(track, path) for track, path in matched if path is not None]
print("local audio files:", len(audio_index))
print("matched English Pop tracks:", len(matched))
assert matched, "DALI ID와 일치하는 로컬 음원이 없습니다. 음원 파일명을 <DALI_ID>.<ext>로 맞추세요."

In [ ]:
from IPython.display import Audio, display

sample_track, sample_audio_path = matched[0]
print(sample_track.dali_id, sample_track.artist, "-", sample_track.title)
display(Audio(filename=str(sample_audio_path)))

## 4. HTDemucs 6-stem으로 한 곡 분리

In [ ]:
from mohim_dali.separator import StemSeparator

separator = StemSeparator(device=DEVICE, model_name="htdemucs_6s")
stems, sample_rate, mixture = separator.separate(sample_audio_path)
print("sample rate:", sample_rate)
print("stems:", list(stems))

In [ ]:
for stem_name in ("vocals", "guitar", "piano", "bass", "other"):
    if stem_name in stems:
        print("---", stem_name, "---")
        display(Audio(stems[stem_name].numpy(), rate=sample_rate))

## 5. Beat This와 반복 4마디 모티프 확인

In [ ]:
import urllib.request

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        "https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt",
        BEAT_CHECKPOINT,
    )
print(BEAT_CHECKPOINT, BEAT_CHECKPOINT.stat().st_size, "bytes")

In [ ]:
from mohim_dali.motif import MotifConfig, MotifExtractor, create_beat_tracker

beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_config = MotifConfig(
    bars=MOTIF_BARS,
    search_seconds=MOTIF_SEARCH_SECONDS,
    similarity_threshold=MOTIF_SIMILARITY,
)
motif_extractor = MotifExtractor(beat_tracker, motif_config)
motif_result = motif_extractor.extract(sample_audio_path, stems, mixture, sample_rate)

print("selected stem:", motif_result["stem_name"])
print("segment:", motif_result["start_sec"], "~", motif_result["end_sec"])
print("repeat similarity:", motif_result["similarity"])
display(pd.DataFrame(motif_result["stem_scores"]).T.sort_values("total", ascending=False))
display(Audio(motif_result["audio"].numpy(), rate=sample_rate))

## 6. 데이터셋 배치 생성

기본값은 3곡입니다. 결과를 확인한 뒤 설정 셀의 `MAX_SONGS = None`으로 바꾸면 전체를 처리합니다. 완료된 곡은 재실행할 때 건너뜁니다.

In [ ]:
from dataclasses import asdict
from mohim_dali.dataset import DatasetBuilder

builder = DatasetBuilder(
    audio_dir=DALI_AUDIO_DIR,
    output_dir=OUTPUT_DIR,
    separator=separator,
    motif_extractor=motif_extractor,
    audio_format=AUDIO_FORMAT,
    resume=True,
)
matched_tracks = [track for track, _ in matched]
results = builder.build(matched_tracks, max_songs=MAX_SONGS)
results_df = pd.DataFrame([asdict(result) for result in results])
display(results_df)
display(results_df.groupby(["status", "reason"], dropna=False).size().rename("count").reset_index())

## 7. ACE-Step dual-stream manifest 생성

In [ ]:
import json
from mohim_dali.manifest import build_dual_stream_manifest

manifest = build_dual_stream_manifest(OUTPUT_DIR, MANIFEST_PATH)
print("usable samples:", manifest["metadata"]["num_samples"])
print("manifest:", MANIFEST_PATH)
if manifest["samples"]:
    print(json.dumps(manifest["samples"][0], ensure_ascii=False, indent=2)[:3000])

## 8. 공식 ACE-Step clone 및 MOHIM 패치 적용

공식 ACE-Step을 호환 커밋으로 고정한 뒤 이 저장소의 dual-stream 패치를 적용합니다. 패키지 설치는 한 번만 실행하세요.

In [ ]:
from mohim_dali.trainer import (
    DEFAULT_REVISION,
    apply_acestep_patch,
    ensure_acestep_repo,
    install_acestep,
)

ACESTEP_DIR = Path("/content/ACE-Step-1.5")
PATCH_FILE = REPO_DIR / "patches/ace-step-1.5-dual-stream.patch"
ACESTEP_DIR = ensure_acestep_repo(ACESTEP_DIR, revision=DEFAULT_REVISION)
apply_acestep_patch(ACESTEP_DIR, PATCH_FILE)
INSTALL_ACESTEP = True
if INSTALL_ACESTEP:
    install_acestep(ACESTEP_DIR)
print("ACE-Step directory:", ACESTEP_DIR)

## 9. Dual-stream tensor 전처리

In [ ]:
from mohim_dali.trainer import preprocess_dual_stream

CHECKPOINT_DIR = Path("/content/drive/MyDrive/MOHIM/checkpoints")
TENSOR_DIR = Path("/content/drive/MyDrive/MOHIM/dali_dual_tensors")
MODEL_VARIANT = "base"
MAX_DURATION = 240.0

preprocess_dual_stream(
    repo_dir=ACESTEP_DIR,
    manifest_path=MANIFEST_PATH,
    checkpoint_dir=CHECKPOINT_DIR,
    tensor_dir=TENSOR_DIR,
    model_variant=MODEL_VARIANT,
    max_duration=MAX_DURATION,
    device=DEVICE,
    precision="bf16",
)

In [ ]:
import torch

tensor_files = sorted(TENSOR_DIR.glob("*.pt"))
assert tensor_files, "전처리 tensor가 생성되지 않았습니다."
item = torch.load(tensor_files[0], map_location="cpu", weights_only=True)
required = [
    "motif_seed_latents", "motif_seed_attention_mask",
    "motif_target_latents", "motif_target_attention_mask",
    "vocal_target_latents", "vocal_target_attention_mask",
    "encoder_hidden_states", "encoder_attention_mask",
]
for key in required:
    assert key in item, f"missing tensor: {key}"
    print(key, tuple(item[key].shape))

## 10. LoRA 학습

3곡 smoke test에서는 `EPOCHS = 1`로 실행하세요. 전체 데이터셋이 준비된 뒤 epoch와 gradient accumulation을 늘립니다.

In [ ]:
from mohim_dali.trainer import train_lora

LORA_OUTPUT_DIR = Path("/content/drive/MyDrive/MOHIM/dali_lora_run")
EPOCHS = 1 if MAX_SONGS is not None else 10

train_lora(
    repo_dir=ACESTEP_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    tensor_dir=TENSOR_DIR,
    output_dir=LORA_OUTPUT_DIR,
    model_variant=MODEL_VARIANT,
    rank=8,
    alpha=16,
    dropout=0.05,
    batch_size=1,
    gradient_accumulation=4,
    epochs=EPOCHS,
    learning_rate=1e-4,
    save_every=1,
    device=DEVICE,
    precision="bf16",
)

In [ ]:
checkpoints = sorted(LORA_OUTPUT_DIR.rglob("*"))
print("output files:", len(checkpoints))
for path in checkpoints[-30:]:
    if path.is_file():
        print(path.relative_to(LORA_OUTPUT_DIR), path.stat().st_size)